In [ ]:
import numpy as np
import matplotlib.pyplot as plt

epsilon_p = 0.1
E_J = 37 * 2 * np.pi
phi_a = 0.11
phi_b = 0.204
alpha_sq = 8
g = np.sin(epsilon_p) * E_J * phi_a**2 * phi_b
kappa_b = 5 / 10.4
kappa_2 = 4 * g**2 / kappa_b
kappa_a = 0.5 * kappa_2
eps_d = 2 * alpha_sq * g

A_all   = np.arange(-5.2, 5.2, 0.01)
PA_all  = np.arange(-5.2, 5.2, 0.01)
A, PA   = np.meshgrid(A_all, PA_all)

A_0   = np.sqrt(eps_d / g)
delta = kappa_a * kappa_b / 4 / g**2

H_vals = (-PA**2 * kappa_a / 2
          + ((A_0**2 - delta) * A * PA - A**3 * PA - A * PA**3 - 2 * A**2 * PA**2)
          * 2 * g**2 / kappa_b)

plt.figure()
plt.pcolormesh(A_all, PA_all, H_vals, cmap='RdBu_r')
plt.colorbar(label='Hamiltonian')
cs = plt.contour(A_all, PA_all, H_vals, levels=[0.0], colors='white', linewidths=2)
plt.scatter(np.sqrt(A_0**2 - delta), 0, color='black', s=10)
plt.xlabel('A')
plt.ylabel(r'$P_A$')
plt.tight_layout()
plt.show()

In [ ]:
paths = cs.get_paths()
contour_coords = np.concatenate([p.vertices for p in paths], axis=0)  # shape (N, 2)

A_contour  = contour_coords[:, 0]
PA_contour = contour_coords[:, 1]

tol=1e-12
idx_keep = idx_keep = (A_contour > tol) & (PA_contour > tol)

A_keep = A_contour[idx_keep]
PA_keep = PA_contour[idx_keep]

A_0_p = np.sqrt(A_0**2-delta)
A_keep = np.insert(A_keep,0,A_0_p)
A_keep = np.append(A_keep,0)

PA_keep = np.insert(PA_keep,0,0)
PA_keep = np.append(PA_keep,0)

PB_keep = -2*np.sqrt(2)*g/kappa_b*A_keep*PA_keep
B_keep = np.sqrt(2)*g/kappa_b * (A_keep**2+4*A_keep*PA_keep+PA_keep**2-A_0_p**2) - kappa_a/2/np.sqrt(2)/g

fig, axes = plt.subplots(1,2)
ax_1, ax_2 = axes

ax_1.scatter(A_keep,PA_keep, c=range(len(A_keep)))
ax_1.set_xlabel(r'$A$')
ax_1.set_ylabel(r'$P_A$')

sc = ax_2.scatter(B_keep, PB_keep, c=range(len(B_keep)))
ax_2.set_xlabel(r'$B$')
ax_2.set_ylabel(r'$P_B$')

plt.colorbar(sc, ax=ax_2)
plt.tight_layout()
plt.show()

In [ ]:
dA = np.diff(A_keep)
dPA = np.diff(PA_keep)
dB = np.diff(B_keep)
dPB = np.diff(PB_keep)

action = -np.sum(dA*PA_keep[1:]-dPA*A_keep[1:]+dB*PB_keep[1:]-dPB*B_keep[1:])

# Sweep $\kappa_a$

In [ ]:
kappa_a_ratio_lim = 4*g**2*A_0**2/kappa_b/kappa_a
#kappa_a_list = np.logspace(np.log10(kappa_a_ratio_lim)-1e-1,-4,20)[:]*kappa_a
kappa_a_list = np.linspace(kappa_a_ratio_lim-5,10**(-4),100)[:]*kappa_a
action_list=[]
action_A_only_list=[]
plot_path=False
for kappa_a_i in kappa_a_list:
        delta = kappa_a_i * kappa_b / 4 / g**2

        H_vals = (-PA**2 * kappa_a_i / 2
                + ((A_0**2 - delta) * A * PA - A**3 * PA - A * PA**3 - 2 * A**2 * PA**2)
                * 2 * g**2 / kappa_b)

        fig_dummy, ax_dummy = plt.subplots()
        cs = ax_dummy.contour(A_all, PA_all, H_vals, levels=[0.0])
        plt.close(fig_dummy)
        paths = cs.get_paths()
        contour_coords = np.concatenate([p.vertices for p in paths], axis=0)  # shape (N, 2)

        A_contour  = contour_coords[:, 0]
        PA_contour = contour_coords[:, 1]
        idx_keep = idx_keep = (A_contour > tol) & (PA_contour > tol)
        A_keep = A_contour[idx_keep]
        PA_keep = PA_contour[idx_keep]

        A_0_p = np.sqrt(A_0**2-delta)
        A_keep = np.insert(A_keep,0,A_0_p)
        A_keep = np.append(A_keep,0)

        PA_keep = np.insert(PA_keep,0,0)
        PA_keep = np.append(PA_keep,0)

        PB_keep = -2*np.sqrt(2)*g/kappa_b*A_keep*PA_keep
        B_keep = np.sqrt(2)*g/kappa_b * (A_keep**2+4*A_keep*PA_keep+PA_keep**2-A_0_p**2) - kappa_a_i/2/np.sqrt(2)/g
        dA = np.diff(A_keep)
        dPA = np.diff(PA_keep)
        dB = np.diff(B_keep)
        dPB = np.diff(PB_keep)

        action = -np.sum(dA*PA_keep[1:]-dPA*A_keep[1:]+dB*PB_keep[1:]-dPB*B_keep[1:])
        action_list.append(action)
        action_A_only = -np.sum(dA*PA_keep[1:]-dPA*A_keep[1:])
        action_A_only_list.append(action_A_only)

        if plot_path:
                fig, axes = plt.subplots(1,2)
                ax_1, ax_2 = axes

                ax_1.scatter(A_keep,PA_keep, c=range(len(A_keep)))
                ax_1.set_xlabel(r'$A$')
                ax_1.set_ylabel(r'$P_A$')

                sc = ax_2.scatter(B_keep, PB_keep, c=range(len(B_keep)))
                ax_2.set_xlabel(r'$B$')
                ax_2.set_ylabel(r'$P_B$')

                plt.colorbar(sc, ax=ax_2)
                plt.tight_layout()
                plt.show()

kappa_a_list=np.insert(kappa_a_list,0,kappa_a_ratio_lim*kappa_a)

action_list = np.array(action_list)
action_A_only_list = np.array(action_A_only_list)

action_list=np.insert(action_list,0,0)
action_A_only_list=np.insert(action_A_only_list,0,0)

In [ ]:
action_diabatic_list = np.array([12.89359256, 11.04023434,  7.47938626,  4.16774918,  1.07694377,
        0.18829547, 12.89359256, 13.11678039, 13.5941508 , 14.1238747 ,
       14.72927553, 15.07550804, 15.26539729, 15.30500198, 15.34541232,
       15.38653895, 15.42846306, 15.47125586])
kappa_diabatic_list = np.array([0.01365336, 0.02730671, 0.06826679, 0.13653357, 0.27306715,
       0.36864065, 0.01365336, 0.01228802, 0.00955735, 0.00682668,
       0.00409601, 0.00273067, 0.002048  , 0.00191147, 0.00177494,
       0.0016384 , 0.00150187, 0.00136534])

In [ ]:
delta_keep = kappa_a_list*kappa_b/4/g**2
plt.plot(kappa_a_list/2/np.pi*1e3, -action_list, label="Adiabatic limit")
plt.plot(kappa_a_list/2/np.pi*1e3, -action_A_only_list, label="Mode a only")
plt.scatter([0],-A_0**2)
sc = plt.scatter(kappa_diabatic_list/2/np.pi*1e3, -action_diabatic_list, c= kappa_b/(4*np.sqrt(2)*g*np.sqrt(A_0**2-kappa_diabatic_list*kappa_b/4/g**2)))

kappa_a_th = np.linspace(0,kappa_a_ratio_lim*kappa_a)
A_0_p_sq_th = A_0**2 - kappa_a_th*kappa_b/4/g**2
plt.plot(kappa_a_th/2/np.pi*1e3, -A_0_p_sq_th, ls='--', label=r"$-A_0'^2$")

plt.xlabel(r"$\kappa_a/2\pi$"+" (MHz)")
plt.ylabel(r"$S$")
plt.colorbar(sc, label=r"Adiabtic ratio")
#plt.xscale("log")
#plt.xlim(1e-5,1e-1)
#plt.ylim(-16.5,-15.5)
plt.legend()
plt.show()

# Sweep $\alpha^2$

In [ ]:
delta = kappa_a * kappa_b / 4 / g**2
A_0_sq_lim = kappa_a/kappa_2
#kappa_a_list = np.logspace(np.log10(kappa_a_ratio_lim)-1e-1,-4,20)[:]*kappa_a
A_0_sq_list = np.linspace(A_0_sq_lim+.1,16,100)[:]
action_list=[]
action_A_only_list=[]
plot_path=False
for A_0_sq_i in A_0_sq_list:  
        A_0_i = np.sqrt(A_0_sq_i)
        A_all   = np.arange(-.1, 1.1*A_0_i, 0.01)
        PA_all  = np.arange(-.1, 2*A_0_i, 0.01)
        A, PA   = np.meshgrid(A_all, PA_all)     
        H_vals = (-PA**2 * kappa_a / 2
                + ((A_0_i**2 - delta) * A * PA - A**3 * PA - A * PA**3 - 2 * A**2 * PA**2)
                * 2 * g**2 / kappa_b)

        fig_dummy, ax_dummy = plt.subplots()
        cs = ax_dummy.contour(A_all, PA_all, H_vals, levels=[0.0])
        plt.close(fig_dummy)
        paths = cs.get_paths()
        contour_coords = np.concatenate([p.vertices for p in paths], axis=0)  # shape (N, 2)

        A_contour  = contour_coords[:, 0]
        PA_contour = contour_coords[:, 1]
        idx_keep = idx_keep = (A_contour > tol) & (PA_contour > tol)
        A_keep = A_contour[idx_keep]
        PA_keep = PA_contour[idx_keep]

        A_0_p = np.sqrt(A_0_i**2-delta)
        A_keep = np.insert(A_keep,0,A_0_p)
        A_keep = np.append(A_keep,0)

        PA_keep = np.insert(PA_keep,0,0)
        PA_keep = np.append(PA_keep,0)

        PB_keep = -2*np.sqrt(2)*g/kappa_b*A_keep*PA_keep
        B_keep = np.sqrt(2)*g/kappa_b * (A_keep**2+4*A_keep*PA_keep+PA_keep**2-A_0_p**2) - kappa_a_i/2/np.sqrt(2)/g
        dA = np.diff(A_keep)
        dPA = np.diff(PA_keep)
        dB = np.diff(B_keep)
        dPB = np.diff(PB_keep)

        action = -np.sum(dA*PA_keep[1:]-dPA*A_keep[1:]+dB*PB_keep[1:]-dPB*B_keep[1:])
        action_list.append(action)
        action_A_only = -np.sum(dA*PA_keep[1:]-dPA*A_keep[1:])
        action_A_only_list.append(action_A_only)

        if plot_path:
                fig, axes = plt.subplots(1,2)
                ax_1, ax_2 = axes

                ax_1.scatter(A_keep,PA_keep, c=range(len(A_keep)))
                ax_1.set_xlabel(r'$A$')
                ax_1.set_ylabel(r'$P_A$')

                sc = ax_2.scatter(B_keep, PB_keep, c=range(len(B_keep)))
                ax_2.set_xlabel(r'$B$')
                ax_2.set_ylabel(r'$P_B$')

                plt.colorbar(sc, ax=ax_2)
                plt.tight_layout()
                plt.show()

action_list = np.array(action_list)
action_A_only_list = np.array(action_A_only_list)

In [ ]:
action_diabatic_list = np.array([12.88933668, 11.44459473,  8.57768153,  5.75367941,  3.00742718,
        2.34359297,  1.69694771,  1.31903872])

A_0_sq_diabatic_list = np.array([16.  , 14.4 , 11.2 ,  8.  ,  4.8 ,  4.  ,  3.2 ,  2.72])

In [ ]:

plt.plot(A_0_sq_list, -action_list, label="Adiabatic limit")
plt.plot(A_0_sq_list, -action_A_only_list, label="Mode a only")


sc = plt.scatter(A_0_sq_diabatic_list, -action_diabatic_list, c=kappa_b/(4*np.sqrt(2)*g*np.sqrt(A_0_sq_diabatic_list-delta)))
plt.colorbar(sc, label=r"Adiabtic ratio")

A_0_p_sq_th = A_0_sq_list - kappa_a*kappa_b/4/g**2
plt.plot(A_0_sq_list, -A_0_p_sq_th, ls='--', label=r"$-A_0'^2$")

plt.xlabel(r"$A_0^2$"+" (MHz)")
plt.ylabel(r"$S$")
plt.legend()
#plt.xlim(.4,3)
#plt.ylim(-3,.1)
plt.show()

# Sweep $\kappa_b$

In [ ]:
kappa_b_ratio_lim = 4*g**2*A_0**2/kappa_a/kappa_b
#kappa_a_list = np.logspace(np.log10(kappa_a_ratio_lim)-1e-1,-4,20)[:]*kappa_a
kappa_b_list = np.linspace(kappa_b_ratio_lim-5,10**(-4),100)[:]*kappa_b
action_list=[]
action_A_only_list=[]
plot_path=False
for kappa_b_i in kappa_b_list:
        delta = kappa_b_i * kappa_a / 4 / g**2

        H_vals = (-PA**2 * kappa_a / 2
                + ((A_0**2 - delta) * A * PA - A**3 * PA - A * PA**3 - 2 * A**2 * PA**2)
                * 2 * g**2 / kappa_b_i)

        fig_dummy, ax_dummy = plt.subplots()
        cs = ax_dummy.contour(A_all, PA_all, H_vals, levels=[0.0])
        plt.close(fig_dummy)
        paths = cs.get_paths()
        contour_coords = np.concatenate([p.vertices for p in paths], axis=0)  # shape (N, 2)

        A_contour  = contour_coords[:, 0]
        PA_contour = contour_coords[:, 1]
        idx_keep = idx_keep = (A_contour > tol) & (PA_contour > tol)
        A_keep = A_contour[idx_keep]
        PA_keep = PA_contour[idx_keep]

        A_0_p = np.sqrt(A_0**2-delta)
        A_keep = np.insert(A_keep,0,A_0_p)
        A_keep = np.append(A_keep,0)

        PA_keep = np.insert(PA_keep,0,0)
        PA_keep = np.append(PA_keep,0)

        PB_keep = -2*np.sqrt(2)*g/kappa_b_i*A_keep*PA_keep
        B_keep = np.sqrt(2)*g/kappa_b_i * (A_keep**2+4*A_keep*PA_keep+PA_keep**2-A_0_p**2) - kappa_a/2/np.sqrt(2)/g
        dA = np.diff(A_keep)
        dPA = np.diff(PA_keep)
        dB = np.diff(B_keep)
        dPB = np.diff(PB_keep)

        action = -np.sum(dA*PA_keep[1:]-dPA*A_keep[1:]+dB*PB_keep[1:]-dPB*B_keep[1:])
        action_list.append(action)
        action_A_only = -np.sum(dA*PA_keep[1:]-dPA*A_keep[1:])
        action_A_only_list.append(action_A_only)

        if plot_path:
                fig, axes = plt.subplots(1,2)
                ax_1, ax_2 = axes

                ax_1.scatter(A_keep,PA_keep, c=range(len(A_keep)))
                ax_1.set_xlabel(r'$A$')
                ax_1.set_ylabel(r'$P_A$')

                sc = ax_2.scatter(B_keep, PB_keep, c=range(len(B_keep)))
                ax_2.set_xlabel(r'$B$')
                ax_2.set_ylabel(r'$P_B$')

                plt.colorbar(sc, ax=ax_2)
                plt.tight_layout()
                plt.show()

action_list = np.array(action_list)
action_A_only_list = np.array(action_A_only_list)

In [ ]:
action_diabatic_list = np.array([12.89359201, 12.68928942, 12.47708021, 12.25999638, 12.0401578 ,
       11.81916573, 11.59801492, 12.89359201, 12.9913371 , 13.17266232,
       13.31740933, 13.34790448])
kappa_b_diabatic_list = np.array([1. , 1.2, 1.4, 1.6, 1.8, 2. , 2.2, 1. , 0.9, 0.7, 0.5, 0.3])*kappa_b

In [ ]:
plt.plot(kappa_b_list[:-1]/2/np.pi*1e3, -action_list[:-1], label="Adiabatic limit")
plt.plot(kappa_b_list/2/np.pi*1e3, -action_A_only_list, label="Mode a only")

sc = plt.scatter(kappa_b_diabatic_list/2/np.pi*1e3, -action_diabatic_list, c= kappa_b_diabatic_list/(4*np.sqrt(2)*g*np.sqrt(A_0**2-kappa_a*kappa_b_diabatic_list/4/g**2)))
plt.colorbar(sc, label=r"Adiabtic ratio")

kappa_b_list_th = np.linspace(0, kappa_b_ratio_lim*kappa_b)
A_0_p_sq_th = A_0**2 - kappa_a*kappa_b_list_th/4/g**2
plt.plot(kappa_b_list_th/2/np.pi*1e3, -A_0_p_sq_th, ls='--', label=r"$-A_0'^2$")

plt.xlabel(r"$\kappa_b/2\pi$"+" (MHz)")
plt.ylabel(r"$S$")
plt.legend()
plt.show()

# Same but matches pertubative approach

In [ ]:
epsilon_p = 0.1
E_J = 37 * 2 * np.pi
phi_a = 0.11
phi_b = 0.204
alpha_sq = 8
g = np.sin(epsilon_p) * E_J * phi_a**2 * phi_b
kappa_b = 10 / 10.4
kappa_2 = 4 * g**2 / kappa_b
kappa_a = 0.005 * kappa_2
eps_d = 2 * alpha_sq * g

A_all   = np.arange(-5.2, 5.2, 0.01)
PA_all  = np.arange(-5.2, 5.2, 0.01)
A, PA   = np.meshgrid(A_all, PA_all)

A_0   = np.sqrt(eps_d / g)
delta = kappa_a * kappa_b / 4 / g**2

In [ ]:
kappa_a_ratio_lim = 4*g**2*A_0**2/kappa_b/kappa_a
#kappa_a_list = np.logspace(np.log10(kappa_a_ratio_lim)-1e-1,-4,20)[:]*kappa_a
kappa_a_list = np.linspace(kappa_a_ratio_lim-5,10**(-4),100)[:]*kappa_a
action_list=[]
action_A_only_list=[]
plot_path=False
for kappa_a_i in kappa_a_list:
        delta = kappa_a_i * kappa_b / 4 / g**2

        H_vals = (-PA**2 * kappa_a_i / 2
                + ((A_0**2 - delta) * A * PA - A**3 * PA - A * PA**3 - 2 * A**2 * PA**2)
                * 2 * g**2 / kappa_b)

        fig_dummy, ax_dummy = plt.subplots()
        cs = ax_dummy.contour(A_all, PA_all, H_vals, levels=[0.0])
        plt.close(fig_dummy)
        paths = cs.get_paths()
        contour_coords = np.concatenate([p.vertices for p in paths], axis=0)  # shape (N, 2)

        A_contour  = contour_coords[:, 0]
        PA_contour = contour_coords[:, 1]
        idx_keep = idx_keep = (A_contour > tol) & (PA_contour > tol)
        A_keep = A_contour[idx_keep]
        PA_keep = PA_contour[idx_keep]

        A_0_p = np.sqrt(A_0**2-delta)
        A_keep = np.insert(A_keep,0,A_0_p)
        A_keep = np.append(A_keep,0)

        PA_keep = np.insert(PA_keep,0,0)
        PA_keep = np.append(PA_keep,0)

        PB_keep = -2*np.sqrt(2)*g/kappa_b*A_keep*PA_keep
        B_keep = np.sqrt(2)*g/kappa_b * (A_keep**2+4*A_keep*PA_keep+PA_keep**2-A_0_p**2) - kappa_a_i/2/np.sqrt(2)/g
        dA = np.diff(A_keep)
        dPA = np.diff(PA_keep)
        dB = np.diff(B_keep)
        dPB = np.diff(PB_keep)

        action = -np.sum(dA*PA_keep[1:]-dPA*A_keep[1:]+dB*PB_keep[1:]-dPB*B_keep[1:])
        action_list.append(action)
        action_A_only = -np.sum(dA*PA_keep[1:]-dPA*A_keep[1:])
        action_A_only_list.append(action_A_only)

        if plot_path:
                fig, axes = plt.subplots(1,2)
                ax_1, ax_2 = axes

                ax_1.scatter(A_keep,PA_keep, c=range(len(A_keep)))
                ax_1.set_xlabel(r'$A$')
                ax_1.set_ylabel(r'$P_A$')

                sc = ax_2.scatter(B_keep, PB_keep, c=range(len(B_keep)))
                ax_2.set_xlabel(r'$B$')
                ax_2.set_ylabel(r'$P_B$')

                plt.colorbar(sc, ax=ax_2)
                plt.tight_layout()
                plt.show()

kappa_a_list=np.insert(kappa_a_list,0,kappa_a_ratio_lim*kappa_a)

action_list = np.array(action_list)
action_A_only_list = np.array(action_A_only_list)

action_list=np.insert(action_list,0,0)
action_A_only_list=np.insert(action_A_only_list,0,0)

In [ ]:
delta_keep = kappa_a_list*kappa_b/4/g**2
plt.plot(kappa_a_list/2/np.pi*1e3, -action_list, label="Adiabatic limit")
plt.plot(kappa_a_list/2/np.pi*1e3, -action_A_only_list, label="Mode a only")
plt.scatter([0],-A_0**2)

kappa_a_th = np.linspace(0,kappa_a_ratio_lim*kappa_a)
A_0_p_sq_th = A_0**2 - kappa_a_th*kappa_b/4/g**2
plt.plot(kappa_a_th/2/np.pi*1e3, -A_0_p_sq_th, ls='--', label=r"$-A_0'^2$")

plt.xlabel(r"$\kappa_a/2\pi$"+" (MHz)")
plt.ylabel(r"$S$")
plt.legend()
plt.show()

In [ ]:
delta = kappa_a * kappa_b / 4 / g**2
A_0_sq_lim = kappa_a/kappa_2
#kappa_a_list = np.logspace(np.log10(kappa_a_ratio_lim)-1e-1,-4,20)[:]*kappa_a
A_0_sq_list = np.linspace(A_0_sq_lim+.1,100,300)[:]
action_list=[]
action_A_only_list=[]
plot_path=False
for A_0_sq_i in A_0_sq_list:  
        A_0_i = np.sqrt(A_0_sq_i)
        A_all   = np.arange(-.1, 1.1*A_0_i, 0.01)
        PA_all  = np.arange(-.1, 2*A_0_i, 0.01)
        A, PA   = np.meshgrid(A_all, PA_all)     
        H_vals = (-PA**2 * kappa_a / 2
                + ((A_0_i**2 - delta) * A * PA - A**3 * PA - A * PA**3 - 2 * A**2 * PA**2)
                * 2 * g**2 / kappa_b)

        fig_dummy, ax_dummy = plt.subplots()
        cs = ax_dummy.contour(A_all, PA_all, H_vals, levels=[0.0])
        plt.close(fig_dummy)
        paths = cs.get_paths()
        contour_coords = np.concatenate([p.vertices for p in paths], axis=0)  # shape (N, 2)

        A_contour  = contour_coords[:, 0]
        PA_contour = contour_coords[:, 1]
        idx_keep = idx_keep = (A_contour > tol) & (PA_contour > tol)
        A_keep = A_contour[idx_keep]
        PA_keep = PA_contour[idx_keep]

        A_0_p = np.sqrt(A_0_i**2-delta)
        A_keep = np.insert(A_keep,0,A_0_p)
        A_keep = np.append(A_keep,0)

        PA_keep = np.insert(PA_keep,0,0)
        PA_keep = np.append(PA_keep,0)

        PB_keep = -2*np.sqrt(2)*g/kappa_b*A_keep*PA_keep
        B_keep = np.sqrt(2)*g/kappa_b * (A_keep**2+4*A_keep*PA_keep+PA_keep**2-A_0_p**2) - kappa_a_i/2/np.sqrt(2)/g
        dA = np.diff(A_keep)
        dPA = np.diff(PA_keep)
        dB = np.diff(B_keep)
        dPB = np.diff(PB_keep)

        action = -np.sum(dA*PA_keep[1:]-dPA*A_keep[1:]+dB*PB_keep[1:]-dPB*B_keep[1:])
        action_list.append(action)
        action_A_only = -np.sum(dA*PA_keep[1:]-dPA*A_keep[1:])
        action_A_only_list.append(action_A_only)

        if plot_path:
                fig, axes = plt.subplots(1,2)
                ax_1, ax_2 = axes

                ax_1.scatter(A_keep,PA_keep, c=range(len(A_keep)))
                ax_1.set_xlabel(r'$A$')
                ax_1.set_ylabel(r'$P_A$')

                sc = ax_2.scatter(B_keep, PB_keep, c=range(len(B_keep)))
                ax_2.set_xlabel(r'$B$')
                ax_2.set_ylabel(r'$P_B$')

                plt.colorbar(sc, ax=ax_2)
                plt.tight_layout()
                plt.show()


action_list = np.array(action_list)
action_A_only_list = np.array(action_A_only_list)

In [ ]:

plt.plot(A_0_sq_list, -action_list, label="Adiabatic limit")
plt.plot(A_0_sq_list, -action_A_only_list, label="Mode a only")

A_0_p_sq_th = A_0_sq_list - kappa_a*kappa_b/4/g**2
plt.plot(A_0_sq_list, -A_0_p_sq_th, ls='--', label=r"$-A_0'^2$")

plt.xlabel(r"$A_0^2$")
plt.ylabel(r"$S$")
plt.legend()
plt.show()

In [ ]:
plt.plot(A_0_sq_list, (A_0_p_sq_th-action_list)/action_list)
plt.xlabel(r"$A_0^2$")
plt.ylabel(r"$|S-A_0^2|/S$")
plt.show()

In [ ]:
kappa_b_ratio_lim = 4*g**2*A_0**2/kappa_a/kappa_b
#kappa_a_list = np.logspace(np.log10(kappa_a_ratio_lim)-1e-1,-4,20)[:]*kappa_a
kappa_b_list = np.linspace(kappa_b_ratio_lim-5,10**(-4),100)[:]*kappa_b
action_list=[]
action_A_only_list=[]
plot_path=False
for kappa_b_i in kappa_b_list:
        delta = kappa_b_i * kappa_a / 4 / g**2

        H_vals = (-PA**2 * kappa_a / 2
                + ((A_0**2 - delta) * A * PA - A**3 * PA - A * PA**3 - 2 * A**2 * PA**2)
                * 2 * g**2 / kappa_b_i)

        fig_dummy, ax_dummy = plt.subplots()
        cs = ax_dummy.contour(A_all, PA_all, H_vals, levels=[0.0])
        plt.close(fig_dummy)
        paths = cs.get_paths()
        contour_coords = np.concatenate([p.vertices for p in paths], axis=0)  # shape (N, 2)

        A_contour  = contour_coords[:, 0]
        PA_contour = contour_coords[:, 1]
        idx_keep = idx_keep = (A_contour > tol) & (PA_contour > tol)
        A_keep = A_contour[idx_keep]
        PA_keep = PA_contour[idx_keep]

        A_0_p = np.sqrt(A_0**2-delta)
        A_keep = np.insert(A_keep,0,A_0_p)
        A_keep = np.append(A_keep,0)

        PA_keep = np.insert(PA_keep,0,0)
        PA_keep = np.append(PA_keep,0)

        PB_keep = -2*np.sqrt(2)*g/kappa_b_i*A_keep*PA_keep
        B_keep = np.sqrt(2)*g/kappa_b_i * (A_keep**2+4*A_keep*PA_keep+PA_keep**2-A_0_p**2) - kappa_a/2/np.sqrt(2)/g
        dA = np.diff(A_keep)
        dPA = np.diff(PA_keep)
        dB = np.diff(B_keep)
        dPB = np.diff(PB_keep)

        action = -np.sum(dA*PA_keep[1:]-dPA*A_keep[1:]+dB*PB_keep[1:]-dPB*B_keep[1:])
        action_list.append(action)
        action_A_only = -np.sum(dA*PA_keep[1:]-dPA*A_keep[1:])
        action_A_only_list.append(action_A_only)

        if plot_path:
                fig, axes = plt.subplots(1,2)
                ax_1, ax_2 = axes

                ax_1.scatter(A_keep,PA_keep, c=range(len(A_keep)))
                ax_1.set_xlabel(r'$A$')
                ax_1.set_ylabel(r'$P_A$')

                sc = ax_2.scatter(B_keep, PB_keep, c=range(len(B_keep)))
                ax_2.set_xlabel(r'$B$')
                ax_2.set_ylabel(r'$P_B$')

                plt.colorbar(sc, ax=ax_2)
                plt.tight_layout()
                plt.show()

action_list = np.array(action_list)
action_A_only_list = np.array(action_A_only_list)


In [ ]:
plt.plot(kappa_b_list[:-1]/2/np.pi*1e3, -action_list[:-1], label="Adiabatic limit")
plt.plot(kappa_b_list/2/np.pi*1e3, -action_A_only_list, label="Mode a only")

kappa_b_list_th = np.linspace(0, kappa_b_ratio_lim*kappa_b)
A_0_p_sq_th = A_0**2 - kappa_a*kappa_b_list_th/4/g**2
plt.plot(kappa_b_list_th/2/np.pi*1e3, -A_0_p_sq_th, ls='--', label=r"$-A_0'^2$")

plt.xlabel(r"$\kappa_b/2\pi$"+" (MHz)")
plt.ylabel(r"$S$")
plt.legend()
plt.show()